# Sparse Matrix–Vector Multiplication Using CSR in CUDA C

## Jupyter / Google Colab Laboratory Notebook

This notebook implements **Sparse Matrix–Vector Multiplication (SpMV)** using **CSR (Compressed Sparse Row)** representation and CUDA C.

### Learning objectives
- Understand CSR sparse matrix representation.
- Transfer CSR data from CPU to GPU.
- Launch a CUDA kernel with one thread per matrix row.
- Perform parallel sparse matrix-vector multiplication.
- Understand why CSR can avoid the `atomicAdd()` used in the COO version.


## 1. Sparse Matrix

We use the following 4×4 sparse matrix:

```text
A =
[5  0  0  8]
[0  3  0  0]
[0  0  6  0]
[2  0  0  7]
```

Input vector:

```text
X = [1 2 3 4]
```

The matrix contains **NNZ = 6** non-zero elements.

## 2. CSR Representation

CSR uses three arrays:

```text
values     = {5, 8, 3, 6, 2, 7}
col_index  = {0, 3, 1, 2, 0, 3}
row_ptr    = {0, 2, 3, 4, 6}
```

`row_ptr[r]` and `row_ptr[r+1]` identify the non-zero values belonging to row `r`.

```text
Row 0 → indices 0,1
Row 1 → index  2
Row 2 → index  3
Row 3 → indices 4,5
```


## 3. Why One CUDA Thread per Row?

In CSR SpMV, each row can calculate its own dot product independently:

```text
Thread 0 → calculates Y[0]
Thread 1 → calculates Y[1]
Thread 2 → calculates Y[2]
Thread 3 → calculates Y[3]
```

Because each thread writes to a different `y[row]`, the basic implementation does **not require `atomicAdd()`**.


In [ ]:
!nvcc --version


## 4. Complete CUDA C Program

The following program performs CSR sparse matrix-vector multiplication on the GPU.

In [ ]:
%%writefile sparse_csr.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define ROWS 4
#define COLS 4
#define NNZ 6

// CUDA Kernel: one thread processes one matrix row
__global__ void sparseMatVecCSR(
    int *values,
    int *col_index,
    int *row_ptr,
    int *x,
    int *y)
{
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < ROWS)
    {
        int sum = 0;

        int start = row_ptr[row];
        int end = row_ptr[row + 1];

        for (int j = start; j < end; j++)
        {
            sum += values[j] * x[col_index[j]];
        }

        y[row] = sum;
    }
}

int main()
{
    // CSR representation on CPU
    int h_values[NNZ] = {5, 8, 3, 6, 2, 7};
    int h_col_index[NNZ] = {0, 3, 1, 2, 0, 3};
    int h_row_ptr[ROWS + 1] = {0, 2, 3, 4, 6};

    // Input vector
    int h_x[COLS] = {1, 2, 3, 4};

    // Output vector
    int h_y[ROWS] = {0};

    // Device pointers
    int *d_values, *d_col_index, *d_row_ptr;
    int *d_x, *d_y;

    printf("===============================================\n");
    printf(" Sparse Matrix-Vector Multiplication in CUDA\n");
    printf(" CSR Representation\n");
    printf("===============================================\n\n");

    printf("CSR Values: ");
    for (int i = 0; i < NNZ; i++)
        printf("%d ", h_values[i]);

    printf("\nColumn Indices: ");
    for (int i = 0; i < NNZ; i++)
        printf("%d ", h_col_index[i]);

    printf("\nRow Pointer: ");
    for (int i = 0; i < ROWS + 1; i++)
        printf("%d ", h_row_ptr[i]);

    printf("\nInput Vector: ");
    for (int i = 0; i < COLS; i++)
        printf("%d ", h_x[i]);

    // Allocate GPU memory
    cudaMalloc((void**)&d_values, NNZ * sizeof(int));
    cudaMalloc((void**)&d_col_index, NNZ * sizeof(int));
    cudaMalloc((void**)&d_row_ptr, (ROWS + 1) * sizeof(int));
    cudaMalloc((void**)&d_x, COLS * sizeof(int));
    cudaMalloc((void**)&d_y, ROWS * sizeof(int));

    // Copy data CPU → GPU
    cudaMemcpy(d_values, h_values, NNZ * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_col_index, h_col_index, NNZ * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_row_ptr, h_row_ptr, (ROWS + 1) * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_x, h_x, COLS * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemset(d_y, 0, ROWS * sizeof(int));

    // Launch enough threads for all rows
    int threadsPerBlock = 256;
    int blocksPerGrid = (ROWS + threadsPerBlock - 1) / threadsPerBlock;

    printf("\n\nLaunching CUDA Kernel...\n");
    printf("Threads per block = %d\n", threadsPerBlock);
    printf("Blocks = %d\n", blocksPerGrid);

    sparseMatVecCSR<<<blocksPerGrid, threadsPerBlock>>>(
        d_values,
        d_col_index,
        d_row_ptr,
        d_x,
        d_y);

    cudaError_t error = cudaGetLastError();
    if (error != cudaSuccess)
    {
        printf("Kernel launch error: %s\n", cudaGetErrorString(error));
        return 1;
    }

    cudaDeviceSynchronize();

    // Copy result GPU → CPU
    cudaMemcpy(h_y, d_y, ROWS * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nOutput Vector:\n");
    for (int i = 0; i < ROWS; i++)
        printf("Y[%d] = %d\n", i, h_y[i]);

    // Free GPU memory
    cudaFree(d_values);
    cudaFree(d_col_index);
    cudaFree(d_row_ptr);
    cudaFree(d_x);
    cudaFree(d_y);

    printf("\nProgram completed successfully.\n");

    return 0;
}


In [ ]:
!nvcc sparse_csr.cu -o sparse_csr_cuda
!./sparse_csr_cuda


## 5. Expected Output

```text
CSR Values: 5 8 3 6 2 7
Column Indices: 0 3 1 2 0 3
Row Pointer: 0 2 3 4 6
Input Vector: 1 2 3 4

Output Vector:
Y[0] = 37
Y[1] = 6
Y[2] = 18
Y[3] = 30
```


## 6. Kernel Explanation

### Thread identification
```c
int row = blockIdx.x * blockDim.x + threadIdx.x;
```
This calculates the global thread ID and uses it as the matrix row.

### Row boundaries
```c
int start = row_ptr[row];
int end = row_ptr[row + 1];
```
These identify the CSR entries belonging to the row.

### Dot product
```c
for (int j = start; j < end; j++)
    sum += values[j] * x[col_index[j]];
```
Only non-zero matrix elements are processed.

### Store result
```c
y[row] = sum;
```
Each thread writes to its own output element, so no `atomicAdd()` is needed in this basic row-parallel CSR implementation.

## 7. Step-by-Step GPU Execution

```text
CPU
 │
 ├── values[]
 ├── col_index[]
 ├── row_ptr[]
 └── X
       │
       │ cudaMemcpy
       ▼
GPU Global Memory
       │
       ▼
CUDA Threads
       │
       ├── Thread 0 → Row 0 → Y[0] = 37
       ├── Thread 1 → Row 1 → Y[1] = 6
       ├── Thread 2 → Row 2 → Y[2] = 18
       └── Thread 3 → Row 3 → Y[3] = 30
       │
       │ cudaMemcpy
       ▼
CPU Output
Y = [37, 6, 18, 30]
```


## 8. COO vs CSR in CUDA

| Feature | COO | CSR |
|---|---|---|
| Row storage | `row[]` | `row_ptr[]` |
| Column storage | `col[]` | `col_index[]` |
| Values | `val[]` | `values[]` |
| Basic parallel strategy | One thread per non-zero | One thread per row |
| `atomicAdd()` | Often required when rows share output | Not required for one-thread-per-row basic SpMV |
| Main advantage | Simple representation | Efficient row-wise computation |


## 9. Important CUDA Parameters

### Kernel launch
```cpp
sparseMatVecCSR<<<blocksPerGrid, threadsPerBlock>>>(...);
```

`threadsPerBlock` specifies the number of CUDA threads in each block.

`blocksPerGrid` specifies the number of blocks.

For a larger matrix, the same formula works:

```cpp
int blocksPerGrid = (ROWS + threadsPerBlock - 1) / threadsPerBlock;
```

The condition `row < ROWS` prevents extra threads from accessing invalid rows.

## 10. Laboratory Exercises

1. Change the matrix to a 5×5 sparse matrix.
2. Change `NNZ` and update the CSR arrays.
3. Add CUDA error checking after every CUDA API call.
4. Compare execution of COO and CSR implementations.
5. Implement a dense matrix-vector multiplication and compare it with CSR.
6. Measure execution time using CUDA events.
7. Implement a larger CSR matrix and experiment with different block sizes such as 32, 64, 128, and 256.
